# Local Windows LoRA run

Run this notebook with a native Windows Python kernel from this repository. The environment must already contain the project dependencies, `git` must be on `PATH`, and an NVIDIA CUDA GPU is required. Do not run these cells in WSL.

The notebook stores adapters under `local_artifacts/`, results under `local_results/`, and the external RAG clone under `dms-rag/`. These folders are ignored by Git.

In [ ]:
import os
import sys
from pathlib import Path

# Set deterministic CUDA configuration before importing torch or the workflow.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "jura_hypersumm").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook from inside the JURA_hypersumm repository.")

REPO_ROOT = find_repository_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
TRAIN_PATH = REPO_ROOT / "train_ternary.csv"
VAL_PATH = REPO_ROOT / "val_ternary.csv"
RAG_DIR = REPO_ROOT / "dms-rag"
ARTIFACT_ROOT = REPO_ROOT / "local_artifacts"
RESULTS_DIR = REPO_ROOT / "local_results"
DOCX_DIR = REPO_ROOT / "local_docx"
DOCX_DIR.mkdir(parents=True, exist_ok=True)

# Put private test decisions in local_docx/. They are read in place and are not deleted.
DOCUMENT_PATHS = sorted(DOCX_DIR.glob("*.docx"))

# Change to True after a completed run to reuse the saved adapter without training.
USE_EXISTING_MODEL = False

print(f"Repository: {REPO_ROOT}")
print(f"DOCX files found: {len(DOCUMENT_PATHS)}")

Ministral normally does not require a Hugging Face token. For a gated model such as Llama, set `HF_TOKEN` in the Windows environment before starting Jupyter; do not paste a token into this notebook. If `DOCUMENT_PATHS` is empty, training and validation run normally and document testing is skipped.

In [ ]:
from jura_hypersumm.lora import run as _run_lora

def run(model_name, task, hyperparameters=None, **overrides):
    """Run the repository LoRA workflow with native Windows paths."""
    local_options = {
        "train_path": TRAIN_PATH,
        "val_path": VAL_PATH,
        "rag_dir": RAG_DIR,
        "drive_root": ARTIFACT_ROOT,
        "results_dir": RESULTS_DIR,
        "document_paths": DOCUMENT_PATHS,
        "use_existing_model": USE_EXISTING_MODEL,
    }
    local_options.update(overrides)
    return _run_lora(model_name, task, hyperparameters, **local_options)

In [ ]:
scores = run("ministral", "ternary")
scores